In [ ]:
# Mount Google Drive and initialize feature extraction paths

from google.colab import drive
from pathlib import Path

import json
import random
import hashlib
import time
import joblib
import platform

import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data' / 'processed'
SPLITS_DIR = PROJECT_ROOT / '01_data' / 'splits'
FEATURES_DIR = PROJECT_ROOT / '01_data' / 'features'

CLASSICAL_DIR = PROJECT_ROOT / '02_notebooks' / '03_classical_ml'
NOTEBOOK_DIR = CLASSICAL_DIR / '00_feature_extraction'

RESULTS_DIR = NOTEBOOK_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'

VECTORIZERS_DIR = (
    PROJECT_ROOT / '04_models' / 'classical' / 'vectorizers'
)

assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

for folder in [
    FEATURES_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    VECTORIZERS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

# Load shared project configuration

with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']

random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid')

# Record environment versions

environment_df = pd.DataFrame([
    {'Package': 'Python', 'Version': platform.python_version()},
    {'Package': 'NumPy', 'Version': np.__version__},
    {'Package': 'Pandas', 'Version': pd.__version__},
    {'Package': 'SciPy', 'Version': scipy.__version__},
    {'Package': 'Scikit-learn', 'Version': sklearn.__version__},
    {'Package': 'Joblib', 'Version': joblib.__version__}
])

environment_df.to_csv(
    TABLES_DIR / 'environment_versions.csv',
    index=False
)

display(environment_df)

print("\nNotebook:", NOTEBOOK_DIR.name)
print("Processed datasets:", PROCESSED_DIR)
print("Feature storage:", FEATURES_DIR)
print("Vectorizer storage:", VECTORIZERS_DIR)
print("Results directory:", RESULTS_DIR)
print("Target labels:", LABELS)
print("Random seed:", SEED)
print("\nFeature extraction environment initialized.")

In [ ]:
# Load and verify the representative training and validation datasets

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

def calculate_sha256(path):
    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            sha256.update(chunk)

    return sha256.hexdigest()


datasets = {}
verification_records = []

selected_partitions = [
    'train_representative',
    'validation'
]

for partition in selected_partitions:

    record = dataset_manifest.loc[
        dataset_manifest['Partition'] == partition
    ].iloc[0]

    path = PROJECT_ROOT / record['Dataset Path']

    actual_hash = calculate_sha256(path)

    assert actual_hash == record['SHA256'], \
        f"Fingerprint mismatch: {partition}"

    df = pd.read_parquet(path)

    assert len(df) == record['Rows']
    assert df['id'].is_unique
    assert df['comment_text'].notna().all()
    assert df['comment_text'].str.strip().ne('').all()
    assert df[LABELS].isin([0, 1]).all().all()

    # Verify partition membership and row order
    ids = pd.read_csv(
        PROJECT_ROOT / record['Manifest Path'],
        dtype={'id': str}
    )

    assert df['id'].tolist() == ids['id'].tolist()
    assert df['group_id'].tolist() == ids['group_id'].tolist()
    assert ids['row_index'].tolist() == list(range(len(df)))

    datasets[partition] = df

    verification_records.append({
        'Partition': partition,
        'Comments': len(df),
        'Unique IDs': df['id'].nunique(),
        'SHA256 Verified': True,
        'Row Alignment Verified': True,
        'Valid Labels': True,
        'Missing Comments': df['comment_text'].isna().sum()
    })

train_df = datasets['train_representative']
val_df = datasets['validation']

# Verify independence of training and validation partitions

assert set(train_df['id']).isdisjoint(val_df['id'])
assert set(train_df['group_id']).isdisjoint(val_df['group_id'])

# Separate text inputs and target labels

X_train_text = train_df['comment_text'].tolist()
X_val_text = val_df['comment_text'].tolist()

y_train = train_df[LABELS].to_numpy(dtype=np.int8)
y_val = val_df[LABELS].to_numpy(dtype=np.int8)

verification_df = pd.DataFrame(verification_records)

display(verification_df)

verification_df.to_csv(
    TABLES_DIR / 'dataset_verification.csv',
    index=False
)

# Verify positive label support

label_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': y_train.sum(axis=0),
    'Validation Positives': y_val.sum(axis=0),
    'Training Prevalence (%)': y_train.mean(axis=0) * 100,
    'Validation Prevalence (%)': y_val.mean(axis=0) * 100
})

display(label_support_df.round(4))

label_support_df.to_csv(
    TABLES_DIR / 'dataset_label_support.csv',
    index=False
)

print("\nTraining comments:", len(X_train_text))
print("Validation comments:", len(X_val_text))
print("Training label shape:", y_train.shape)
print("Validation label shape:", y_val.shape)
print("Training-validation ID overlap: 0")
print("Training-validation group overlap: 0")
print("Official test dataset: Not loaded")
print("\nDataset verification completed successfully.")

In [ ]:
# Analyse word vocabulary and document frequency using training data only

from sklearn.feature_extraction.text import CountVectorizer
import gc

WORD_NGRAM_RANGE = (1, 2)
WORD_TOKEN_PATTERN = r'(?u)\b\w+\b'

MIN_DF_CANDIDATES = [1, 2, 3, 5, 10]
FEATURE_BUDGETS = [25000, 50000, 100000]

start_time = time.perf_counter()

word_probe = CountVectorizer(
    lowercase=True,
    token_pattern=WORD_TOKEN_PATTERN,
    ngram_range=WORD_NGRAM_RANGE,
    stop_words=None,
    min_df=1,
    dtype=np.int32
)

word_counts = word_probe.fit_transform(X_train_text).tocsr()

feature_names = word_probe.get_feature_names_out()

# Calculate document frequency and total occurrence frequency

document_frequency = np.bincount(
    word_counts.indices,
    minlength=len(feature_names)
)

term_frequency = np.bincount(
    word_counts.indices,
    weights=word_counts.data,
    minlength=len(feature_names)
)

ngram_orders = np.array([
    term.count(' ') + 1 for term in feature_names
])

total_occurrences = term_frequency.sum()

# Examine the effect of minimum document frequency

vocabulary_records = []

for min_df in MIN_DF_CANDIDATES:

    eligible = document_frequency >= min_df

    vocabulary_records.append({
        'Minimum Document Frequency': min_df,
        'Unigrams': np.sum(eligible & (ngram_orders == 1)),
        'Bigrams': np.sum(eligible & (ngram_orders == 2)),
        'Total Features': eligible.sum(),
        'Vocabulary Retained (%)':
            eligible.mean() * 100,
        'Term Occurrences Retained (%)':
            term_frequency[eligible].sum()
            / total_occurrences * 100
    })

vocabulary_df = pd.DataFrame(vocabulary_records)

print("Vocabulary and document frequency analysis")
display(vocabulary_df.round(2))

vocabulary_df.to_csv(
    TABLES_DIR / 'word_vocabulary_analysis.csv',
    index=False
)

# Examine candidate feature budgets after frequency filtering

budget_records = []

for min_df in MIN_DF_CANDIDATES:

    eligible = document_frequency >= min_df

    ranked_frequencies = np.sort(
        term_frequency[eligible]
    )[::-1]

    for budget in FEATURE_BUDGETS:

        selected = ranked_frequencies[:budget]

        budget_records.append({
            'Minimum Document Frequency': min_df,
            'Feature Budget': budget,
            'Available Features': len(ranked_frequencies),
            'Selected Features': len(selected),
            'Total Occurrences Retained (%)':
                selected.sum() / total_occurrences * 100,
            'Eligible Occurrences Retained (%)':
                selected.sum() / ranked_frequencies.sum() * 100
        })

word_budget_df = pd.DataFrame(budget_records)

print("\nFeature budget analysis")
display(word_budget_df.round(2))

word_budget_df.to_csv(
    TABLES_DIR / 'word_feature_budget_analysis.csv',
    index=False
)

print("\nTotal unique word features:", len(feature_names))
print("Unique unigrams:", np.sum(ngram_orders == 1))
print("Unique bigrams:", np.sum(ngram_orders == 2))
print("Features appearing in only one document:",
      np.sum(document_frequency == 1))

print("Analysis time (seconds):",
      round(time.perf_counter() - start_time, 2))

# Release temporary vocabulary analysis objects

del word_counts, word_probe, feature_names
del document_frequency, term_frequency, ngram_orders
gc.collect()

print("\nWord vocabulary analysis completed.")

In [ ]:
# Extract word-level TF-IDF features using the representative training subset

FEATURE_BUDGET = 50000

WORD_TFIDF_CONFIG = {
    'analyzer': 'word',
    'ngram_range': (1, 2),
    'token_pattern': WORD_TOKEN_PATTERN,
    'min_df': 2,
    'max_df': 1.0,
    'max_features': FEATURE_BUDGET,
    'lowercase': True,
    'stop_words': None,
    'sublinear_tf': True,
    'use_idf': True,
    'smooth_idf': True,
    'norm': 'l2',
    'dtype': np.float32
}

word_vectorizer = TfidfVectorizer(**WORD_TFIDF_CONFIG)

# Fit exclusively on training comments

start_time = time.perf_counter()

X_train_word = word_vectorizer.fit_transform(
    X_train_text
).tocsr()

word_fit_time = time.perf_counter() - start_time

# Transform validation comments using the fitted vocabulary

start_time = time.perf_counter()

X_val_word = word_vectorizer.transform(
    X_val_text
).tocsr()

word_transform_time = time.perf_counter() - start_time

# Verify feature dimensions and numerical validity

assert X_train_word.shape[0] == len(y_train)
assert X_val_word.shape[0] == len(y_val)

assert X_train_word.shape[1] == X_val_word.shape[1]
assert X_train_word.shape[1] <= FEATURE_BUDGET

assert np.isfinite(X_train_word.data).all()
assert np.isfinite(X_val_word.data).all()

assert (X_train_word.data >= 0).all()
assert (X_val_word.data >= 0).all()

# Summarize sparse feature matrices

word_feature_records = []

for partition, matrix in [
    ('Training', X_train_word),
    ('Validation', X_val_word)
]:

    total_elements = matrix.shape[0] * matrix.shape[1]

    memory_bytes = (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    )

    nonzero_per_row = np.diff(matrix.indptr)

    word_feature_records.append({
        'Representation': 'Word TF-IDF',
        'Partition': partition,
        'Comments': matrix.shape[0],
        'Features': matrix.shape[1],
        'Nonzero Elements': matrix.nnz,
        'Density (%)': matrix.nnz / total_elements * 100,
        'Sparsity (%)': (1 - matrix.nnz / total_elements) * 100,
        'Mean Nonzero Features': nonzero_per_row.mean(),
        'Median Nonzero Features': np.median(nonzero_per_row),
        'Zero Feature Comments': np.sum(nonzero_per_row == 0),
        'Sparse Memory (MB)': memory_bytes / (1024 ** 2),
        'Data Type': str(matrix.dtype)
    })

word_feature_summary_df = pd.DataFrame(
    word_feature_records
)

display(word_feature_summary_df.round(4))

word_feature_summary_df.to_csv(
    TABLES_DIR / 'word_tfidf_feature_summary.csv',
    index=False
)

# Save vectorizer configuration and extraction times

word_config = WORD_TFIDF_CONFIG.copy()
word_config['dtype'] = 'float32'

word_config['actual_features'] = X_train_word.shape[1]
word_config['training_samples'] = len(X_train_text)
word_config['fit_time_seconds'] = word_fit_time
word_config['validation_transform_time_seconds'] = (
    word_transform_time
)

with open(
    RUN_CONFIGS_DIR / 'word_tfidf_config.json', 'w'
) as file:
    json.dump(word_config, file, indent=4)

print("\nTraining feature shape:", X_train_word.shape)
print("Validation feature shape:", X_val_word.shape)

print("Vocabulary size:", len(word_vectorizer.vocabulary_))

print("Training fit time (seconds):", round(word_fit_time, 2))
print("Validation transform time (seconds):",
      round(word_transform_time, 2))

print("Data type:", X_train_word.dtype)
print("Feature format:", X_train_word.format)

print("\nWord TF-IDF extraction completed successfully.")

In [ ]:
# Analyse the fitted word TF-IDF vocabulary and feature distributions

word_features = word_vectorizer.get_feature_names_out()
word_idf = word_vectorizer.idf_

word_ngram_orders = np.array([
    term.count(' ') + 1 for term in word_features
])

word_vocabulary_df = pd.DataFrame({
    'Feature': word_features,
    'Ngram Order': word_ngram_orders,
    'IDF': word_idf
})

word_vocabulary_df.to_csv(
    TABLES_DIR / 'word_tfidf_vocabulary.csv',
    index=False
)

# Analyse unigram and bigram composition

word_composition_df = (
    word_vocabulary_df
    .groupby('Ngram Order')
    .agg(
        Features=('Feature', 'count'),
        Mean_IDF=('IDF', 'mean'),
        Median_IDF=('IDF', 'median'),
        Minimum_IDF=('IDF', 'min'),
        Maximum_IDF=('IDF', 'max')
    )
    .reset_index()
)

word_composition_df['Feature Share (%)'] = (
    word_composition_df['Features']
    / len(word_features) * 100
)

print("Word vocabulary composition")
display(word_composition_df.round(4))

word_composition_df.to_csv(
    TABLES_DIR / 'word_tfidf_vocabulary_composition.csv',
    index=False
)

# Inspect features with the lowest IDF values

frequent_word_features_df = (
    word_vocabulary_df
    .sort_values('IDF')
    .head(20)
    .reset_index(drop=True)
)

print("\nFeatures with the lowest IDF values")
display(frequent_word_features_df.round(4))

frequent_word_features_df.to_csv(
    TABLES_DIR / 'word_tfidf_lowest_idf_features.csv',
    index=False
)

# Identify comments with no recognized word features

zero_feature_records = []
zero_feature_summaries = []

for partition, matrix, df in [
    ('Training', X_train_word, train_df),
    ('Validation', X_val_word, val_df)
]:

    nonzero_counts = np.diff(matrix.indptr)
    zero_mask = nonzero_counts == 0

    zero_df = df.loc[
        zero_mask,
        ['id', 'comment_text'] + LABELS
    ].copy()

    zero_df.insert(0, 'Partition', partition)

    zero_feature_records.append(zero_df)

    zero_feature_summaries.append({
        'Partition': partition,
        'Total Comments': len(df),
        'Zero Feature Comments': len(zero_df),
        'Zero Feature Rate (%)':
            len(zero_df) / len(df) * 100,
        'Clean Comments':
            zero_df[LABELS].sum(axis=1).eq(0).sum(),
        'Toxic Comments':
            zero_df[LABELS].sum(axis=1).gt(0).sum(),
        **{
            label: int(zero_df[label].sum())
            for label in LABELS
        }
    })

zero_word_features_df = pd.concat(
    zero_feature_records,
    ignore_index=True
)

zero_word_summary_df = pd.DataFrame(
    zero_feature_summaries
)

print("\nZero-feature comment summary")
display(zero_word_summary_df.round(4))

zero_word_features_df.to_csv(
    TABLES_DIR / 'word_tfidf_zero_feature_comments.csv',
    index=False
)

zero_word_summary_df.to_csv(
    TABLES_DIR / 'word_tfidf_zero_feature_summary.csv',
    index=False
)

# Visualize vocabulary and sparse feature distributions

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

composition = word_composition_df.set_index(
    'Ngram Order'
)['Features']

bars = axes[0].bar(
    ['Unigrams', 'Bigrams'],
    [composition.get(1, 0), composition.get(2, 0)],
    color=['#4C78A8', '#F58518']
)

axes[0].bar_label(bars, fmt='%.0f', padding=4)
axes[0].set_ylim(0, composition.max() * 1.15)
axes[0].set_title('Vocabulary Composition')
axes[0].set_ylabel('Number of Features')

axes[1].hist(
    word_idf,
    bins=40,
    color='#54A24B',
    edgecolor='white'
)

axes[1].set_title('Inverse Document Frequency Distribution')
axes[1].set_xlabel('IDF Value')
axes[1].set_ylabel('Number of Features')

train_active = np.diff(X_train_word.indptr)
val_active = np.diff(X_val_word.indptr)

upper_limit = int(np.ceil(max(
    np.quantile(train_active, 0.99),
    np.quantile(val_active, 0.99)
) / 50) * 50)

bins = np.linspace(0, upper_limit, 50)

for counts, name, color in [
    (train_active, 'Training', '#4C78A8'),
    (val_active, 'Validation', '#F58518')
]:

    axes[2].hist(
        counts,
        bins=bins,
        weights=np.ones(len(counts)) / len(counts) * 100,
        histtype='step',
        linewidth=2,
        color=color,
        label=name
    )

axes[2].set_title('Active Features per Comment')
axes[2].set_xlabel('Nonzero Features')
axes[2].set_ylabel('Percentage of All Comments')
axes[2].set_xlim(0, upper_limit)
axes[2].legend()

fig.suptitle(
    'Word-Level TF-IDF Feature Analysis',
    fontsize=16
)

fig.text(
    0.5, 0.01,
    f'Active feature distributions displayed up to '
    f'{upper_limit} features for readability.',
    ha='center',
    fontsize=10
)

sns.despine()
plt.tight_layout(rect=[0, 0.05, 1, 0.94])

plt.savefig(
    FIGURES_DIR / 'word_tfidf_feature_analysis.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

print("\nWord TF-IDF feature analysis completed.")

In [ ]:
# Analyse character vocabulary using training data only

CHAR_ORDERS = [2, 3, 4, 5]
CHAR_RANGES = [(2, 4), (3, 5), (2, 5)]
CHAR_MIN_DF = [2, 3, 5]

char_frequency_data = {}
char_vocabulary_records = []

start_time = time.perf_counter()

for n in CHAR_ORDERS:

    print(f"Analysing character {n}-grams...")

    char_probe = CountVectorizer(
        analyzer='char',
        ngram_range=(n, n),
        lowercase=True,
        min_df=1,
        dtype=np.int32
    )

    char_counts = char_probe.fit_transform(
        X_train_text
    ).tocsr()

    vocabulary_size = char_counts.shape[1]

    document_frequency = np.bincount(
        char_counts.indices,
        minlength=vocabulary_size
    )

    term_frequency = np.bincount(
        char_counts.indices,
        weights=char_counts.data,
        minlength=vocabulary_size
    )

    char_frequency_data[n] = {
        'document_frequency': document_frequency,
        'term_frequency': term_frequency
    }

    for min_df in [1] + CHAR_MIN_DF:

        eligible = document_frequency >= min_df

        char_vocabulary_records.append({
            'Ngram Order': n,
            'Minimum Document Frequency': min_df,
            'Original Vocabulary': vocabulary_size,
            'Retained Features': eligible.sum(),
            'Vocabulary Retained (%)':
                eligible.mean() * 100,
            'Term Occurrences Retained (%)':
                term_frequency[eligible].sum()
                / term_frequency.sum() * 100
        })

    print("Unique features:", vocabulary_size)

    del char_counts, char_probe
    gc.collect()

char_vocabulary_df = pd.DataFrame(
    char_vocabulary_records
)

print("\nCharacter vocabulary analysis")
display(char_vocabulary_df.round(2))

char_vocabulary_df.to_csv(
    TABLES_DIR / 'character_vocabulary_analysis.csv',
    index=False
)

# Compare candidate n-gram ranges and feature budgets

char_budget_records = []

for low, high in CHAR_RANGES:

    orders = list(range(low, high + 1))

    total_occurrences = sum(
        char_frequency_data[n]['term_frequency'].sum()
        for n in orders
    )

    for min_df in CHAR_MIN_DF:

        eligible_frequencies = np.concatenate([
            char_frequency_data[n]['term_frequency'][
                char_frequency_data[n]['document_frequency']
                >= min_df
            ]
            for n in orders
        ])

        ranked_frequencies = np.sort(
            eligible_frequencies
        )[::-1]

        for budget in FEATURE_BUDGETS:

            selected = ranked_frequencies[:budget]

            char_budget_records.append({
                'Ngram Range': f'{low}-{high}',
                'Minimum Document Frequency': min_df,
                'Feature Budget': budget,
                'Available Features': len(ranked_frequencies),
                'Selected Features': len(selected),
                'Total Occurrences Retained (%)':
                    selected.sum() / total_occurrences * 100,
                'Eligible Occurrences Retained (%)':
                    selected.sum()
                    / ranked_frequencies.sum() * 100
            })

char_budget_df = pd.DataFrame(char_budget_records)

char_budget_df.to_csv(
    TABLES_DIR / 'character_feature_budget_analysis.csv',
    index=False
)

print("\nComparison at the common 50000-feature budget")

display(
    char_budget_df[
        char_budget_df['Feature Budget'] == 50000
    ].round(2)
)

print("\nFeature budget comparison at min_df=2")

display(
    char_budget_df[
        char_budget_df['Minimum Document Frequency'] == 2
    ].round(2)
)

print("\nAnalysis time (seconds):",
      round(time.perf_counter() - start_time, 2))

# Release temporary frequency analysis objects

del char_frequency_data
del document_frequency, term_frequency
del eligible_frequencies, ranked_frequencies, selected
gc.collect()

print("\nCharacter vocabulary analysis completed.")

In [ ]:
# Extract character-level TF-IDF features using the training subset

CHAR_TFIDF_CONFIG = {
    'analyzer': 'char',
    'ngram_range': (2, 5),
    'min_df': 2,
    'max_df': 1.0,
    'max_features': FEATURE_BUDGET,
    'lowercase': True,
    'sublinear_tf': True,
    'use_idf': True,
    'smooth_idf': True,
    'norm': 'l2',
    'dtype': np.float32
}

char_vectorizer = TfidfVectorizer(**CHAR_TFIDF_CONFIG)

# Fit exclusively on training comments

start_time = time.perf_counter()

X_train_char = char_vectorizer.fit_transform(
    X_train_text
).tocsr()

char_fit_time = time.perf_counter() - start_time

# Transform validation comments using the fitted vocabulary

start_time = time.perf_counter()

X_val_char = char_vectorizer.transform(
    X_val_text
).tocsr()

char_transform_time = time.perf_counter() - start_time

# Verify feature dimensions and numerical validity

assert X_train_char.shape[0] == len(y_train)
assert X_val_char.shape[0] == len(y_val)

assert X_train_char.shape[1] == X_val_char.shape[1]
assert X_train_char.shape[1] <= FEATURE_BUDGET

assert np.isfinite(X_train_char.data).all()
assert np.isfinite(X_val_char.data).all()

assert (X_train_char.data >= 0).all()
assert (X_val_char.data >= 0).all()

# Summarize sparse feature matrices

char_feature_records = []

for partition, matrix in [
    ('Training', X_train_char),
    ('Validation', X_val_char)
]:

    total_elements = matrix.shape[0] * matrix.shape[1]

    memory_bytes = (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    )

    nonzero_per_row = np.diff(matrix.indptr)

    char_feature_records.append({
        'Representation': 'Character TF-IDF',
        'Partition': partition,
        'Comments': matrix.shape[0],
        'Features': matrix.shape[1],
        'Nonzero Elements': matrix.nnz,
        'Density (%)': matrix.nnz / total_elements * 100,
        'Sparsity (%)':
            (1 - matrix.nnz / total_elements) * 100,
        'Mean Nonzero Features': nonzero_per_row.mean(),
        'Median Nonzero Features': np.median(nonzero_per_row),
        'Zero Feature Comments': np.sum(nonzero_per_row == 0),
        'Sparse Memory (MB)': memory_bytes / (1024 ** 2),
        'Data Type': str(matrix.dtype)
    })

char_feature_summary_df = pd.DataFrame(
    char_feature_records
)

display(char_feature_summary_df.round(4))

char_feature_summary_df.to_csv(
    TABLES_DIR / 'character_tfidf_feature_summary.csv',
    index=False
)

# Save vectorizer configuration and extraction times

char_config = CHAR_TFIDF_CONFIG.copy()
char_config['dtype'] = 'float32'

char_config['actual_features'] = X_train_char.shape[1]
char_config['training_samples'] = len(X_train_text)
char_config['fit_time_seconds'] = char_fit_time
char_config['validation_transform_time_seconds'] = (
    char_transform_time
)

with open(
    RUN_CONFIGS_DIR / 'character_tfidf_config.json', 'w'
) as file:
    json.dump(char_config, file, indent=4)

print("\nTraining feature shape:", X_train_char.shape)
print("Validation feature shape:", X_val_char.shape)

print("Vocabulary size:", len(char_vectorizer.vocabulary_))

print("Training fit time (seconds):", round(char_fit_time, 2))
print("Validation transform time (seconds):",
      round(char_transform_time, 2))

print("Data type:", X_train_char.dtype)
print("Feature format:", X_train_char.format)

print("\nCharacter TF-IDF extraction completed successfully.")

In [ ]:
# Analyse character vocabulary and feature distributions

char_features = char_vectorizer.get_feature_names_out()
char_idf = char_vectorizer.idf_

char_ngram_orders = np.array([
    len(term) for term in char_features
])

char_vocabulary_df = pd.DataFrame({
    'Feature': char_features,
    'Ngram Order': char_ngram_orders,
    'IDF': char_idf
})

char_vocabulary_df.to_csv(
    TABLES_DIR / 'character_tfidf_vocabulary.csv',
    index=False
)

# Analyse selected character n-gram composition

char_composition_df = (
    char_vocabulary_df
    .groupby('Ngram Order')
    .agg(
        Features=('Feature', 'count'),
        Mean_IDF=('IDF', 'mean'),
        Median_IDF=('IDF', 'median'),
        Minimum_IDF=('IDF', 'min'),
        Maximum_IDF=('IDF', 'max')
    )
    .reset_index()
)

char_composition_df['Feature Share (%)'] = (
    char_composition_df['Features']
    / len(char_features) * 100
)

print("Character vocabulary composition")
display(char_composition_df.round(4))

char_composition_df.to_csv(
    TABLES_DIR / 'character_tfidf_vocabulary_composition.csv',
    index=False
)

# Identify character features occurring in many documents

frequent_char_features_df = (
    char_vocabulary_df
    .sort_values('IDF')
    .head(20)
    .reset_index(drop=True)
)

frequent_char_features_df['Feature'] = (
    frequent_char_features_df['Feature'].map(repr)
)

print("\nCharacter features with the lowest IDF values")
display(frequent_char_features_df.round(4))

frequent_char_features_df.to_csv(
    TABLES_DIR / 'character_tfidf_lowest_idf_features.csv',
    index=False
)

# Examine recovery of comments with zero word features

recovery_records = []
recovered_comments = []

for partition, word_matrix, char_matrix, df in [
    ('Training', X_train_word, X_train_char, train_df),
    ('Validation', X_val_word, X_val_char, val_df)
]:

    word_active = np.diff(word_matrix.indptr)
    char_active = np.diff(char_matrix.indptr)

    word_zero = word_active == 0
    char_zero = char_active == 0

    recovered = word_zero & (char_active > 0)

    any_toxic = df[LABELS].sum(axis=1).to_numpy() > 0

    recovery_records.append({
        'Partition': partition,
        'Total Comments': len(df),
        'Word Zero Features': word_zero.sum(),
        'Character Zero Features': char_zero.sum(),
        'Recovered Comments': recovered.sum(),
        'Recovered Toxic Comments':
            (recovered & any_toxic).sum(),
        'Recovery Rate (%)':
            recovered.sum() / word_zero.sum() * 100
            if word_zero.any() else np.nan
    })

    recovered_df = df.loc[
        recovered,
        ['id', 'comment_text'] + LABELS
    ].copy()

    recovered_df.insert(0, 'Partition', partition)

    recovered_df['Word Active Features'] = (
        word_active[recovered]
    )

    recovered_df['Character Active Features'] = (
        char_active[recovered]
    )

    recovered_comments.append(recovered_df)

char_recovery_df = pd.DataFrame(recovery_records)

print("\nZero-feature comment recovery")
display(char_recovery_df.round(4))

char_recovery_df.to_csv(
    TABLES_DIR / 'character_feature_recovery_summary.csv',
    index=False
)

pd.concat(
    recovered_comments,
    ignore_index=True
).to_csv(
    TABLES_DIR / 'character_feature_recovered_comments.csv',
    index=False
)

# Visualize character feature characteristics

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

bars = axes[0].bar(
    char_composition_df['Ngram Order'].astype(str),
    char_composition_df['Features'],
    color=['#4C78A8', '#F58518', '#54A24B', '#B279A2']
)

axes[0].bar_label(
    bars,
    fmt='%.0f',
    padding=4
)

axes[0].set_ylim(
    0, char_composition_df['Features'].max() * 1.15
)

axes[0].set_title('Character Vocabulary Composition')
axes[0].set_xlabel('Character N-gram Length')
axes[0].set_ylabel('Number of Features')

axes[1].hist(
    char_idf,
    bins=40,
    color='#54A24B',
    edgecolor='white'
)

axes[1].set_title('Inverse Document Frequency Distribution')
axes[1].set_xlabel('IDF Value')
axes[1].set_ylabel('Number of Features')

train_char_active = np.diff(X_train_char.indptr)
val_char_active = np.diff(X_val_char.indptr)

upper_limit = int(np.ceil(max(
    np.quantile(train_char_active, 0.99),
    np.quantile(val_char_active, 0.99)
) / 500) * 500)

bins = np.linspace(0, upper_limit, 50)

for counts, name, color in [
    (train_char_active, 'Training', '#4C78A8'),
    (val_char_active, 'Validation', '#F58518')
]:

    axes[2].hist(
        counts,
        bins=bins,
        weights=np.ones(len(counts)) / len(counts) * 100,
        histtype='step',
        linewidth=2,
        color=color,
        label=name
    )

axes[2].set_title('Active Character Features per Comment')
axes[2].set_xlabel('Nonzero Features')
axes[2].set_ylabel('Percentage of All Comments')
axes[2].set_xlim(0, upper_limit)
axes[2].legend()

fig.suptitle(
    'Character-Level TF-IDF Feature Analysis',
    fontsize=16
)

fig.text(
    0.5, 0.01,
    f'Active feature distributions displayed up to '
    f'{upper_limit} features for readability.',
    ha='center',
    fontsize=10
)

sns.despine()
plt.tight_layout(rect=[0, 0.05, 1, 0.94])

plt.savefig(
    FIGURES_DIR / 'character_tfidf_feature_analysis.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

print("\nCharacter TF-IDF feature analysis completed.")

In [ ]:
# Construct combined TF-IDF with a controlled total feature budget

from sklearn.preprocessing import normalize

COMBINED_WORD_BUDGET = FEATURE_BUDGET // 2
COMBINED_CHAR_BUDGET = FEATURE_BUDGET - COMBINED_WORD_BUDGET

combined_word_config = WORD_TFIDF_CONFIG.copy()
combined_char_config = CHAR_TFIDF_CONFIG.copy()

combined_word_config['max_features'] = COMBINED_WORD_BUDGET
combined_char_config['max_features'] = COMBINED_CHAR_BUDGET

combined_word_vectorizer = TfidfVectorizer(
    **combined_word_config
)

combined_char_vectorizer = TfidfVectorizer(
    **combined_char_config
)

# Fit both component vectorizers exclusively on training data

start_time = time.perf_counter()

X_train_word_component = (
    combined_word_vectorizer.fit_transform(X_train_text)
    .tocsr()
)

X_train_char_component = (
    combined_char_vectorizer.fit_transform(X_train_text)
    .tocsr()
)

# Concatenate and normalize the combined representation

X_train_combined = sp.hstack(
    [X_train_word_component, X_train_char_component],
    format='csr',
    dtype=np.float32
)

X_train_combined = normalize(
    X_train_combined,
    norm='l2',
    copy=False
)

combined_fit_time = time.perf_counter() - start_time

del X_train_word_component, X_train_char_component
gc.collect()

# Transform validation comments using the fitted vectorizers

start_time = time.perf_counter()

X_val_word_component = (
    combined_word_vectorizer.transform(X_val_text)
    .tocsr()
)

X_val_char_component = (
    combined_char_vectorizer.transform(X_val_text)
    .tocsr()
)

X_val_combined = sp.hstack(
    [X_val_word_component, X_val_char_component],
    format='csr',
    dtype=np.float32
)

X_val_combined = normalize(
    X_val_combined,
    norm='l2',
    copy=False
)

combined_transform_time = time.perf_counter() - start_time

del X_val_word_component, X_val_char_component
gc.collect()

# Verify combined feature dimensions

actual_word_features = len(
    combined_word_vectorizer.vocabulary_
)

actual_char_features = len(
    combined_char_vectorizer.vocabulary_
)

assert X_train_combined.shape[0] == len(y_train)
assert X_val_combined.shape[0] == len(y_val)

assert X_train_combined.shape[1] == (
    actual_word_features + actual_char_features
)

assert X_train_combined.shape[1] == X_val_combined.shape[1]
assert X_train_combined.shape[1] == FEATURE_BUDGET

# Summarize and verify combined sparse matrices

combined_feature_records = []

for partition, matrix in [
    ('Training', X_train_combined),
    ('Validation', X_val_combined)
]:

    assert np.isfinite(matrix.data).all()
    assert (matrix.data >= 0).all()

    total_elements = matrix.shape[0] * matrix.shape[1]

    memory_bytes = (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    )

    nonzero_per_row = np.diff(matrix.indptr)

    row_norms = np.sqrt(
        np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel()
    )

    assert np.allclose(
        row_norms[nonzero_per_row > 0],
        1.0,
        atol=1e-5
    )

    combined_feature_records.append({
        'Representation': 'Combined TF-IDF',
        'Partition': partition,
        'Comments': matrix.shape[0],
        'Features': matrix.shape[1],
        'Nonzero Elements': matrix.nnz,
        'Density (%)': matrix.nnz / total_elements * 100,
        'Sparsity (%)':
            (1 - matrix.nnz / total_elements) * 100,
        'Mean Nonzero Features': nonzero_per_row.mean(),
        'Median Nonzero Features': np.median(nonzero_per_row),
        'Zero Feature Comments': np.sum(nonzero_per_row == 0),
        'Sparse Memory (MB)': memory_bytes / (1024 ** 2),
        'Data Type': str(matrix.dtype)
    })

combined_feature_summary_df = pd.DataFrame(
    combined_feature_records
)

display(combined_feature_summary_df.round(4))

combined_feature_summary_df.to_csv(
    TABLES_DIR / 'combined_tfidf_feature_summary.csv',
    index=False
)

# Save the combined feature configuration

combined_config = {
    'representation': 'combined_tfidf',
    'word_configuration': {
        **combined_word_config,
        'dtype': 'float32'
    },
    'character_configuration': {
        **combined_char_config,
        'dtype': 'float32'
    },
    'word_features': actual_word_features,
    'character_features': actual_char_features,
    'total_features': X_train_combined.shape[1],
    'component_normalization': 'l2',
    'combined_normalization': 'l2',
    'component_weights': {
        'word': 1.0,
        'character': 1.0
    },
    'column_order': ['word', 'character'],
    'training_samples': len(X_train_text),
    'fit_time_seconds': combined_fit_time,
    'validation_transform_time_seconds':
        combined_transform_time
}

with open(
    RUN_CONFIGS_DIR / 'combined_tfidf_config.json', 'w'
) as file:
    json.dump(combined_config, file, indent=4)

print("\nWord component features:", actual_word_features)
print("Character component features:", actual_char_features)

print("Training feature shape:", X_train_combined.shape)
print("Validation feature shape:", X_val_combined.shape)

print("Training extraction time (seconds):",
      round(combined_fit_time, 2))

print("Validation transformation time (seconds):",
      round(combined_transform_time, 2))

print("Data type:", X_train_combined.dtype)
print("Feature format:", X_train_combined.format)

print("Combined L2 normalization: Verified")
print("\nCombined TF-IDF extraction completed successfully.")

In [ ]:
# Compare word, character and combined TF-IDF representations

feature_comparison_df = pd.concat([
    word_feature_summary_df,
    char_feature_summary_df,
    combined_feature_summary_df
], ignore_index=True)

display(feature_comparison_df.round(4))

feature_comparison_df.to_csv(
    TABLES_DIR / 'tfidf_representation_comparison.csv',
    index=False
)

# Compare feature extraction times

feature_time_df = pd.DataFrame([
    {
        'Representation': 'Word TF-IDF',
        'Training Extraction (s)': word_fit_time,
        'Validation Transformation (s)': word_transform_time
    },
    {
        'Representation': 'Character TF-IDF',
        'Training Extraction (s)': char_fit_time,
        'Validation Transformation (s)': char_transform_time
    },
    {
        'Representation': 'Combined TF-IDF',
        'Training Extraction (s)': combined_fit_time,
        'Validation Transformation (s)': combined_transform_time
    }
])

feature_time_df['Total Time (s)'] = (
    feature_time_df['Training Extraction (s)']
    + feature_time_df['Validation Transformation (s)']
)

print("\nFeature extraction time comparison")
display(feature_time_df.round(2))

feature_time_df.to_csv(
    TABLES_DIR / 'tfidf_extraction_time_comparison.csv',
    index=False
)

# Calculate relative computational requirements

train_comparison_df = feature_comparison_df[
    feature_comparison_df['Partition'] == 'Training'
].copy().reset_index(drop=True)

word_memory = train_comparison_df.loc[
    train_comparison_df['Representation'] == 'Word TF-IDF',
    'Sparse Memory (MB)'
].iloc[0]

word_nonzero = train_comparison_df.loc[
    train_comparison_df['Representation'] == 'Word TF-IDF',
    'Nonzero Elements'
].iloc[0]

train_comparison_df['Relative Memory (x)'] = (
    train_comparison_df['Sparse Memory (MB)'] / word_memory
)

train_comparison_df['Relative Nonzero Elements (x)'] = (
    train_comparison_df['Nonzero Elements'] / word_nonzero
)

efficiency_df = train_comparison_df[[
    'Representation',
    'Features',
    'Relative Memory (x)',
    'Relative Nonzero Elements (x)'
]].merge(
    feature_time_df,
    on='Representation'
)

efficiency_df['Relative Total Time (x)'] = (
    efficiency_df['Total Time (s)']
    / feature_time_df.loc[
        feature_time_df['Representation'] == 'Word TF-IDF',
        'Total Time (s)'
    ].iloc[0]
)

print("\nRelative computational requirements")
display(efficiency_df.round(2))

efficiency_df.to_csv(
    TABLES_DIR / 'tfidf_computational_efficiency.csv',
    index=False
)

# Generate consolidated feature comparison figure

REPRESENTATION_ORDER = [
    'Word TF-IDF',
    'Character TF-IDF',
    'Combined TF-IDF'
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

metrics = [
    ('Density (%)', 'Feature Matrix Density', 'Density (%)'),
    ('Mean Nonzero Features', 'Mean Active Features', 'Features per Comment'),
    ('Sparse Memory (MB)', 'Sparse Matrix Memory', 'Memory (MiB)'),
    ('Zero Feature Comments', 'Zero-Feature Comments', 'Number of Comments')
]

for ax, (metric, title, ylabel) in zip(
    axes.flat[:4], metrics
):

    pivot = feature_comparison_df.pivot(
        index='Representation',
        columns='Partition',
        values=metric
    ).reindex(
        REPRESENTATION_ORDER
    )[['Training', 'Validation']]

    pivot.plot(
        kind='bar',
        ax=ax,
        color=['#4C78A8', '#F58518'],
        width=0.7
    )

    for container in ax.containers:

        fmt = '%.3f' if metric == 'Density (%)' else '%.0f'

        if metric == 'Sparse Memory (MB)':
            fmt = '%.2f'

        ax.bar_label(
            container,
            fmt=fmt,
            padding=3,
            fontsize=9
        )

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=0)
    ax.set_ylim(0, max(pivot.to_numpy().max() * 1.2, 1))
    ax.grid(axis='x', visible=False)
    ax.legend(fontsize=9)

# Compare feature extraction and transformation times

time_pivot = feature_time_df.set_index(
    'Representation'
).reindex(REPRESENTATION_ORDER)[[
    'Training Extraction (s)',
    'Validation Transformation (s)'
]]

time_pivot.plot(
    kind='bar',
    ax=axes[1, 1],
    color=['#4C78A8', '#F58518'],
    width=0.7
)

for container in axes[1, 1].containers:
    axes[1, 1].bar_label(
        container,
        fmt='%.1f',
        padding=3,
        fontsize=9
    )

axes[1, 1].set_title('Feature Extraction Time')
axes[1, 1].set_ylabel('Time (seconds)')
axes[1, 1].set_xlabel('')
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].set_ylim(
    0, time_pivot.to_numpy().max() * 1.25
)
axes[1, 1].legend(
    ['Training', 'Validation'],
    fontsize=9
)

# Compare word and character feature allocations

allocation_df = pd.DataFrame({
    'Representation': REPRESENTATION_ORDER,
    'Word Features': [50000, 0, actual_word_features],
    'Character Features': [0, 50000, actual_char_features]
}).set_index('Representation')

allocation_df.plot(
    kind='bar',
    stacked=True,
    ax=axes[1, 2],
    color=['#4C78A8', '#F58518'],
    width=0.6
)

axes[1, 2].set_title('Controlled Feature Allocation')
axes[1, 2].set_ylabel('Number of Features')
axes[1, 2].set_xlabel('')
axes[1, 2].set_ylim(0, FEATURE_BUDGET * 1.2)
axes[1, 2].tick_params(axis='x', rotation=0)
axes[1, 2].legend(fontsize=9)

for ax in axes.flat:
    ax.grid(axis='x', visible=False)

fig.suptitle(
    'Comparative Analysis of TF-IDF Feature Representations',
    fontsize=18
)

sns.despine()
plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.savefig(
    FIGURES_DIR / 'tfidf_representation_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

print("\nTF-IDF comparative analysis completed.")

In [ ]:
# Save sparse feature matrices and fitted TF-IDF vectorizers

feature_matrices = {
    'word_tfidf': {
        'train': X_train_word,
        'validation': X_val_word
    },
    'character_tfidf': {
        'train': X_train_char,
        'validation': X_val_char
    },
    'combined_tfidf': {
        'train': X_train_combined,
        'validation': X_val_combined
    }
}

fitted_vectorizers = {
    'word_tfidf.joblib': word_vectorizer,
    'character_tfidf.joblib': char_vectorizer,
    'combined_word_tfidf.joblib': combined_word_vectorizer,
    'combined_character_tfidf.joblib': combined_char_vectorizer
}

saved_feature_records = []

# Save sparse matrices in compressed NPZ format

for representation, matrices in feature_matrices.items():

    feature_dir = FEATURES_DIR / representation
    feature_dir.mkdir(parents=True, exist_ok=True)

    for partition, matrix in matrices.items():

        path = feature_dir / f'{partition}.npz'

        start_time = time.perf_counter()

        sp.save_npz(
            path,
            matrix,
            compressed=True
        )

        save_time = time.perf_counter() - start_time

        saved_feature_records.append({
            'Representation': representation,
            'Partition': partition,
            'Rows': matrix.shape[0],
            'Features': matrix.shape[1],
            'Nonzero Elements': matrix.nnz,
            'Data Type': str(matrix.dtype),
            'File Size (MiB)': path.stat().st_size / (1024 ** 2),
            'Save Time (s)': save_time,
            'Path': str(path.relative_to(PROJECT_ROOT))
        })

        print(f"Saved: {representation}/{partition}.npz")

saved_features_df = pd.DataFrame(saved_feature_records)

print("\nSaved sparse feature matrices")
display(saved_features_df.round(2))

saved_features_df.to_csv(
    TABLES_DIR / 'saved_feature_matrices.csv',
    index=False
)

# Save fitted vectorizers

saved_vectorizer_records = []

for filename, vectorizer in fitted_vectorizers.items():

    path = VECTORIZERS_DIR / filename

    joblib.dump(
        vectorizer,
        path,
        compress=3
    )

    saved_vectorizer_records.append({
        'Filename': filename,
        'Vocabulary Size': len(vectorizer.vocabulary_),
        'Analyzer': vectorizer.analyzer,
        'Ngram Range': str(vectorizer.ngram_range),
        'File Size (MiB)': path.stat().st_size / (1024 ** 2),
        'Path': str(path.relative_to(PROJECT_ROOT))
    })

    print(f"Saved: {filename}")

saved_vectorizers_df = pd.DataFrame(
    saved_vectorizer_records
)

print("\nSaved vectorizers")
display(saved_vectorizers_df.round(2))

saved_vectorizers_df.to_csv(
    TABLES_DIR / 'saved_vectorizers.csv',
    index=False
)

# Save shared feature row alignment manifests

for partition, df in [
    ('train', train_df),
    ('validation', val_df)
]:

    row_manifest = df[['id', 'group_id']].copy()
    row_manifest.insert(0, 'row_index', np.arange(len(df)))

    row_manifest.to_csv(
        FEATURES_DIR / f'{partition}_row_manifest.csv',
        index=False
    )

    print(f"Saved: {partition}_row_manifest.csv")

print("\nFeature storage summary")

print("Sparse matrices saved:", len(saved_features_df))
print("Fitted vectorizers saved:", len(saved_vectorizers_df))
print("Row alignment manifests saved: 2")

print("Total matrix storage (MiB):",
      round(saved_features_df['File Size (MiB)'].sum(), 2))

print("Total vectorizer storage (MiB):",
      round(saved_vectorizers_df['File Size (MiB)'].sum(), 2))

print("\nFeature artifacts saved successfully.")

In [ ]:
# Reload and verify saved matrices, vectorizers and row manifests

feature_verification_records = []
loaded_vectorizers = {}

# Verify saved sparse feature matrices

for representation, matrices in feature_matrices.items():

    for partition, original in matrices.items():

        path = FEATURES_DIR / representation / f'{partition}.npz'

        loaded = sp.load_npz(path)

        assert loaded.shape == original.shape
        assert loaded.dtype == original.dtype
        assert loaded.format == 'csr'
        assert loaded.nnz == original.nnz

        assert np.array_equal(loaded.indptr, original.indptr)
        assert np.array_equal(loaded.indices, original.indices)
        assert np.array_equal(loaded.data, original.data)

        source_partition = (
            'train_representative'
            if partition == 'train'
            else 'validation'
        )

        source_hash = dataset_manifest.loc[
            dataset_manifest['Partition'] == source_partition,
            'SHA256'
        ].iloc[0]

        feature_verification_records.append({
            'Artifact Type': 'Sparse Matrix',
            'Representation': representation,
            'Partition': partition,
            'Filename': path.name,
            'Rows': loaded.shape[0],
            'Features': loaded.shape[1],
            'Path': str(path.relative_to(PROJECT_ROOT)),
            'SHA256': calculate_sha256(path),
            'Source Dataset SHA256': source_hash,
            'Verification': 'Passed'
        })

        print(f"Verified: {representation}/{partition}.npz")

        del loaded
        gc.collect()

# Verify fitted vectorizers

for filename, original in fitted_vectorizers.items():

    path = VECTORIZERS_DIR / filename

    loaded = joblib.load(path)

    assert loaded.get_params() == original.get_params()
    assert loaded.vocabulary_ == original.vocabulary_
    assert np.array_equal(loaded.idf_, original.idf_)

    loaded_vectorizers[filename] = loaded

    feature_verification_records.append({
        'Artifact Type': 'Vectorizer',
        'Representation': Path(filename).stem,
        'Partition': 'train',
        'Filename': filename,
        'Rows': None,
        'Features': len(loaded.vocabulary_),
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'SHA256': calculate_sha256(path),
        'Source Dataset SHA256': dataset_manifest.loc[
            dataset_manifest['Partition'] == 'train_representative',
            'SHA256'
        ].iloc[0],
        'Verification': 'Passed'
    })

    print(f"Verified: {filename}")

# Verify feature row alignment

for partition, df in [
    ('train', train_df),
    ('validation', val_df)
]:

    path = FEATURES_DIR / f'{partition}_row_manifest.csv'

    row_manifest = pd.read_csv(
        path,
        dtype={'id': str}
    )

    assert len(row_manifest) == len(df)
    assert row_manifest['id'].tolist() == df['id'].tolist()
    assert row_manifest['group_id'].tolist() == df['group_id'].tolist()

    assert np.array_equal(
        row_manifest['row_index'].to_numpy(),
        np.arange(len(df))
    )

    source_partition = (
        'train_representative'
        if partition == 'train'
        else 'validation'
    )

    feature_verification_records.append({
        'Artifact Type': 'Row Manifest',
        'Representation': 'All',
        'Partition': partition,
        'Filename': path.name,
        'Rows': len(row_manifest),
        'Features': None,
        'Path': str(path.relative_to(PROJECT_ROOT)),
        'SHA256': calculate_sha256(path),
        'Source Dataset SHA256': dataset_manifest.loc[
            dataset_manifest['Partition'] == source_partition,
            'SHA256'
        ].iloc[0],
        'Verification': 'Passed'
    })

    print(f"Verified: {path.name}")

# Verify feature reconstruction using reloaded vectorizers

rng = np.random.default_rng(SEED)

reconstruction_records = []

for partition, texts in [
    ('train', X_train_text),
    ('validation', X_val_text)
]:

    random_indices = rng.choice(
        len(texts),
        size=100,
        replace=False
    )

    zero_word_indices = np.flatnonzero(
        np.diff(
            feature_matrices['word_tfidf'][partition].indptr
        ) == 0
    )

    sample_indices = np.unique(
        np.concatenate([random_indices, zero_word_indices])
    )

    sample_texts = [
        texts[i] for i in sample_indices
    ]

    reconstructed_word = loaded_vectorizers[
        'word_tfidf.joblib'
    ].transform(sample_texts).tocsr()

    reconstructed_char = loaded_vectorizers[
        'character_tfidf.joblib'
    ].transform(sample_texts).tocsr()

    combined_word = loaded_vectorizers[
        'combined_word_tfidf.joblib'
    ].transform(sample_texts)

    combined_char = loaded_vectorizers[
        'combined_character_tfidf.joblib'
    ].transform(sample_texts)

    reconstructed_combined = normalize(
        sp.hstack(
            [combined_word, combined_char],
            format='csr',
            dtype=np.float32
        ),
        norm='l2',
        copy=False
    )

    reconstructed_features = {
        'word_tfidf': reconstructed_word,
        'character_tfidf': reconstructed_char,
        'combined_tfidf': reconstructed_combined
    }

    for representation, reconstructed in reconstructed_features.items():

        expected = feature_matrices[
            representation
        ][partition][sample_indices]

        assert reconstructed.shape == expected.shape

        difference = reconstructed - expected

        max_error = (
            float(np.abs(difference.data).max())
            if difference.nnz > 0
            else 0.0
        )

        assert max_error <= 1e-6

        reconstruction_records.append({
            'Representation': representation,
            'Partition': partition,
            'Comments Verified': len(sample_indices),
            'Maximum Absolute Error': max_error,
            'Verification': 'Passed'
        })

# Save verification reports and shared fingerprint registry

feature_verification_df = pd.DataFrame(
    feature_verification_records
)

reconstruction_df = pd.DataFrame(
    reconstruction_records
)

print("\nSaved artifact verification")

display(feature_verification_df[[
    'Artifact Type',
    'Representation',
    'Partition',
    'Filename',
    'Verification'
]])

print("\nFeature reconstruction verification")
display(reconstruction_df)

feature_verification_df.to_csv(
    TABLES_DIR / 'feature_artifact_verification.csv',
    index=False
)

reconstruction_df.to_csv(
    TABLES_DIR / 'feature_reconstruction_verification.csv',
    index=False
)

feature_verification_df.to_csv(
    FEATURES_DIR / 'feature_artifact_manifest.csv',
    index=False
)

print("\nArtifacts verified:", len(feature_verification_df))
print("Reconstruction checks:", len(reconstruction_df))
print("Failed artifact checks: 0")
print("Failed reconstruction checks: 0")
print("Feature artifact fingerprint registry saved.")
print("\nFeature verification completed successfully.")

In [ ]:
# Update the shared feature extraction configuration

with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

config['classical_features'] = {
    'training_partition': 'train_representative',
    'validation_partition': 'validation',
    'training_samples': len(train_df),
    'validation_samples': len(val_df),
    'feature_budget': FEATURE_BUDGET,
    'dtype': 'float32',
    'matrix_format': 'csr',
    'storage_format': 'npz',
    'features_directory': str(
        FEATURES_DIR.relative_to(PROJECT_ROOT)
    ),
    'vectorizers_directory': str(
        VECTORIZERS_DIR.relative_to(PROJECT_ROOT)
    ),
    'artifact_manifest': str(
        (FEATURES_DIR / 'feature_artifact_manifest.csv')
        .relative_to(PROJECT_ROOT)
    ),
    'baseline_representation': 'word_tfidf',
    'representations': {
        'word_tfidf': word_config,
        'character_tfidf': char_config,
        'combined_tfidf': combined_config
    },
    'status': 'verified'
}

with open(CONFIG_DIR / 'project_config.json', 'w') as file:
    json.dump(config, file, indent=4)

print("Project configuration updated.")

# Record methodological decisions

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

decisions_df = pd.read_csv(DECISION_LOG_PATH)

feature_decisions = [
    {
        'decision_id': 'D015',
        'component': 'Word feature extraction',
        'decision': 'Word TF-IDF with unigrams and bigrams',
        'justification': (
            'Captures individual words and local linguistic context. '
            'Stopwords and single-character words are retained. '
            'Stemming and lemmatization are avoided to preserve word forms.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D016',
        'component': 'Vocabulary filtering',
        'decision': 'Minimum document frequency of 2',
        'justification': (
            'Approximately 74 percent of unique word features appear '
            'in only one training document. Filtering retains '
            '88.39 percent of word n-gram occurrences before '
            'applying the feature budget.'
        ),
        'status': 'Initial selection'
    },
    {
        'decision_id': 'D017',
        'component': 'Character feature extraction',
        'decision': 'Character TF-IDF with 2-5 grams',
        'justification': (
            'Captures short and longer character patterns, misspellings '
            'and punctuation. Retains 91.78 percent of character '
            'n-gram occurrences at the selected feature budget. '
            'All 22 word zero-feature comments receive nonzero '
            'character representations.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D018',
        'component': 'Feature dimensionality',
        'decision': 'Common feature budget of 50000',
        'justification': (
            'Word TF-IDF retains 77.13 percent of original word '
            'n-gram occurrences at min_df=2. Equal dimensionality '
            'supports controlled representation comparisons while '
            'limiting computational requirements.'
        ),
        'status': 'Initial selection'
    },
    {
        'decision_id': 'D019',
        'component': 'Combined feature representation',
        'decision': '25000 word and 25000 character features',
        'justification': (
            'Equal initial allocation maintains the same total '
            'dimensionality as individual representations. '
            'Component-wise and final L2 normalization control '
            'the scale of document vectors.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D020',
        'component': 'TF-IDF weighting',
        'decision': 'Sublinear TF, smooth IDF and L2 normalization',
        'justification': (
            'Reduces the influence of highly repetitive terms '
            'and controls differences in document vector magnitude. '
            'Identical weighting settings support controlled comparisons.'
        ),
        'status': 'Initial selection'
    },
    {
        'decision_id': 'D021',
        'component': 'Feature storage and reproducibility',
        'decision': 'Float32 CSR matrices and reusable fitted vectorizers',
        'justification': (
            'Sparse storage avoids unnecessary dense memory allocation. '
            'Six matrices, four vectorizers and two row manifests '
            'passed verification. Sampled feature reconstruction '
            'produced zero numerical error.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D022',
        'component': 'Feature extraction leakage prevention',
        'decision': 'Fit vectorizers exclusively on training comments',
        'justification': (
            'Validation comments are transformed using the vocabulary '
            'and IDF weights learned exclusively from training data. '
            'The official test dataset has not been loaded.'
        ),
        'status': 'Implemented'
    }
]

for record in feature_decisions:

    mask = decisions_df['decision_id'].eq(
        record['decision_id']
    )

    if mask.any():

        for column, value in record.items():
            decisions_df.loc[mask, column] = value

    else:

        decisions_df = pd.concat(
            [decisions_df, pd.DataFrame([record])],
            ignore_index=True
        )

assert decisions_df['decision_id'].is_unique

decisions_df.to_csv(
    DECISION_LOG_PATH,
    index=False
)

print("\nNew feature extraction decisions")

display(
    decisions_df[
        decisions_df['decision_id'].isin(
            [record['decision_id'] for record in feature_decisions]
        )
    ][['decision_id', 'component', 'decision', 'status']]
)

# Verify required notebook artifacts

required_tables = [
    'environment_versions.csv',
    'dataset_verification.csv',
    'dataset_label_support.csv',
    'word_vocabulary_analysis.csv',
    'word_feature_budget_analysis.csv',
    'word_tfidf_feature_summary.csv',
    'word_tfidf_vocabulary.csv',
    'word_tfidf_vocabulary_composition.csv',
    'word_tfidf_lowest_idf_features.csv',
    'word_tfidf_zero_feature_comments.csv',
    'word_tfidf_zero_feature_summary.csv',
    'character_vocabulary_analysis.csv',
    'character_feature_budget_analysis.csv',
    'character_tfidf_feature_summary.csv',
    'character_tfidf_vocabulary.csv',
    'character_tfidf_vocabulary_composition.csv',
    'character_tfidf_lowest_idf_features.csv',
    'character_feature_recovery_summary.csv',
    'character_feature_recovered_comments.csv',
    'combined_tfidf_feature_summary.csv',
    'tfidf_representation_comparison.csv',
    'tfidf_extraction_time_comparison.csv',
    'tfidf_computational_efficiency.csv',
    'saved_feature_matrices.csv',
    'saved_vectorizers.csv',
    'feature_artifact_verification.csv',
    'feature_reconstruction_verification.csv'
]

required_figures = [
    'word_tfidf_feature_analysis.png',
    'character_tfidf_feature_analysis.png',
    'tfidf_representation_comparison.png'
]

required_configs = [
    'word_tfidf_config.json',
    'character_tfidf_config.json',
    'combined_tfidf_config.json'
]

required_shared = [
    PROJECT_ROOT / path
    for path in feature_verification_df['Path']
]

required_shared.append(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

artifact_groups = {
    'Table': [TABLES_DIR / name for name in required_tables],
    'Figure': [FIGURES_DIR / name for name in required_figures],
    'Configuration': [
        RUN_CONFIGS_DIR / name for name in required_configs
    ],
    'Shared Artifact': required_shared
}

final_verification_records = []

for category, paths in artifact_groups.items():

    for path in paths:

        available = path.is_file() and path.stat().st_size > 0

        final_verification_records.append({
            'Type': category,
            'Filename': path.name,
            'Path': str(path.relative_to(PROJECT_ROOT)),
            'Status': 'Available' if available else 'Missing'
        })

final_verification_df = pd.DataFrame(
    final_verification_records
)

final_verification_df.to_csv(
    TABLES_DIR / 'final_feature_artifact_verification.csv',
    index=False
)

assert final_verification_df['Status'].eq('Available').all()
assert feature_verification_df['Verification'].eq('Passed').all()
assert reconstruction_df['Verification'].eq('Passed').all()

print("\nFinal artifact verification")

display(
    final_verification_df
    .groupby(['Type', 'Status'])
    .size()
    .reset_index(name='Artifacts')
)

# Save the feature extraction summary

summary = f"""# Notebook 03.00 - Feature Extraction

## Objective

Construct reusable word, character and combined TF-IDF
representations for controlled classical ML experiments.

## Dataset

- Representative training comments: {len(train_df):,}
- Validation comments: {len(val_df):,}
- Target labels: {len(LABELS)}
- Random seed: {SEED}
- Dataset fingerprints and row alignment verified.
- Official test dataset not loaded.

## Word TF-IDF

- N-gram range: 1-2
- Minimum document frequency: 2
- Vocabulary size: 50,000
- Selected unigrams: 13,456
- Selected bigrams: 36,544
- Lowercase enabled.
- No stopword removal, stemming or lemmatization.

## Character TF-IDF

- Character n-gram range: 2-5
- Minimum document frequency: 2
- Vocabulary size: 50,000
- Character patterns can cross word boundaries.
- All training and validation comments have nonzero features.

## Combined TF-IDF

- Word features: {actual_word_features:,}
- Character features: {actual_char_features:,}
- Total features: {FEATURE_BUDGET:,}
- Equal component weighting.
- Final L2 normalization after concatenation.

## Common Weighting and Storage

- Sublinear term frequency.
- Smooth inverse document frequency.
- L2 normalization.
- Float32 CSR sparse matrices.
- Compressed NPZ storage.

## Computational Findings

Training extraction times:
- Word: {word_fit_time:.2f} seconds
- Character: {char_fit_time:.2f} seconds
- Combined: {combined_fit_time:.2f} seconds

Character and combined representations require approximately
10.81 times the training matrix memory of word TF-IDF.

All 22 comments with zero word features receive nonzero
character representations.

## Saved Artifacts

- Sparse matrices: {len(saved_features_df)}
- Fitted vectorizers: {len(saved_vectorizers_df)}
- Row alignment manifests: 2
- Matrix storage: {saved_features_df['File Size (MiB)'].sum():.2f} MiB
- Vectorizer storage: {saved_vectorizers_df['File Size (MiB)'].sum():.2f} MiB

## Verification

- Artifact integrity checks: 12 passed.
- Sampled reconstruction checks: 6 passed.
- Maximum absolute reconstruction error:
  {reconstruction_df['Maximum Absolute Error'].max():.8f}

## Limitations

- Frequency coverage does not establish predictive usefulness.
- Equal dimensionality does not imply equal computational cost.
- Lowercasing removes capitalization information.
- Vocabulary filtering may remove useful rare expressions.
- Nonzero representation does not guarantee correct classification.
- The selected feature budgets and n-gram ranges are initial
  experimental choices rather than proven optimal configurations.
- Reconstruction was verified on sampled comments, while stored
  matrix equality was verified in full.

## Evaluation Protocol

All five classical baseline models will use identical word
TF-IDF features, training comments and validation comments.

Logistic Regression and Linear SVM will additionally compare
character and combined feature representations.

Full-data experiments must refit vectorizers using the
corresponding full training partition.

The official test dataset remains reserved for final evaluation.

## Conclusion

Feature extraction and verification are complete.
Classification models have not yet been trained.

## Next Stage

Notebook 03.01 - Multinomial Naive Bayes.
"""

(RESULTS_DIR / 'summary.md').write_text(
    summary,
    encoding='utf-8'
)

assert (RESULTS_DIR / 'summary.md').stat().st_size > 0

print("\nFeature extraction completion summary")

print("Analysis tables verified:", len(required_tables))
print("Figures verified:", len(required_figures))
print("Configurations verified:", len(required_configs))
print("Shared artifacts verified:", len(required_shared))
print("Total required artifacts:", len(final_verification_df))

print("Missing artifacts:",
      final_verification_df['Status'].eq('Missing').sum())

print("Total methodological decisions:", len(decisions_df))
print("Notebook summary saved.")
print("\nFeature extraction completed successfully.")